In [3]:
import torch
import subprocess
import os
import shutil
from gradio_client import Client, handle_file

def check_gpu():
    """Check if GPU is available with PyTorch."""
    print("Checking GPU availability with PyTorch...")
    if torch.cuda.is_available():
        print(f"GPU is available: {torch.cuda.get_device_name(0)}")
    else:
        print("GPU is not available.")

def check_whisper():
    """Check if Whisper is installed."""
    print("Checking Whisper installation...")
    try:
        import whisper
        print("Whisper is installed.")
    except ImportError:
        print("Whisper is not installed.")

def check_ffmpeg():
    """Check if ffmpeg is installed."""
    print("Checking ffmpeg installation...")
    try:
        result = subprocess.run(["ffmpeg", "-version"], capture_output=True, text=True)
        if "ffmpeg version" in result.stdout:
            print("ffmpeg is installed.")
        else:
            print("ffmpeg is not installed.")
    except FileNotFoundError:
        print("ffmpeg is not installed.")


In [3]:
!pip install speechbrain torchaudio


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 864.1/864.1 KB 9.3 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.7/37.7 MB 11.4 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 12.1 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 307.7/307.7 KB 11.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.7/117.7 KB 13.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 722.2/722.2 KB 12.6 MB/s eta 0:00:0000:0100:01


In [1]:
#Transfrom the mp3 into wav
!ffmpeg -i ../input/old-testament-scripture-reading-129625.mp3 -ar 16000 -ac 1 ../output/old-testament.wav

ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers
  built with gcc 11 (Ubuntu 11.2.0-19ubuntu1)
  configuration: --prefix=/usr --extra-version=0ubuntu0.22.04.1 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --enable-gnutls --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libgme --enable-libgsm --enable-libjack --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-libpulse --enable-librabbitmq --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libsrt --enable-libssh --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --enable-libvpx --enab

In [ ]:
!ffmpeg -i ../input/conversation_short.mp3 -ar 16000 -ac 1 ../output/speechbrain/conversation_short.wav

ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers
  built with gcc 11 (Ubuntu 11.2.0-19ubuntu1)
  configuration: --prefix=/usr --extra-version=0ubuntu0.22.04.1 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --enable-gnutls --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libgme --enable-libgsm --enable-libjack --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-libpulse --enable-librabbitmq --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libsrt --enable-libssh --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --enable-libvpx --enab

In [ ]:
from speechbrain.inference.separation import SepformerSeparation as separator
import torchaudio
import torch
import subprocess
import os
import shutil
from gradio_client import Client, handle_file

def check_gpu():
    """Check if GPU is available with PyTorch."""
    print("Checking GPU availability with PyTorch...")
    if torch.cuda.is_available():
        print(f"GPU is available: {torch.cuda.get_device_name(0)}")
    else:
        print("GPU is not available.")

def check_whisper():
    """Check if Whisper is installed."""
    print("Checking Whisper installation...")
    try:
        import whisper
        print("Whisper is installed.")
    except ImportError:
        print("Whisper is not installed.")

def check_ffmpeg():
    """Check if ffmpeg is installed."""
    print("Checking ffmpeg installation...")
    try:
        result = subprocess.run(["ffmpeg", "-version"], capture_output=True, text=True)
        if "ffmpeg version" in result.stdout:
            print("ffmpeg is installed.")
        else:
            print("ffmpeg is not installed.")
    except FileNotFoundError:
        print("ffmpeg is not installed.")

# Use GPU if available
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Running on:", device)

# Load model to the correct device
model = separator.from_hparams(
    source="speechbrain/sepformer-whamr",
    savedir="pretrained_models/sepformer-whamr",
    run_opts={"device": device}
)
waveform, sample_rate = torchaudio.load("../output/speechbrain/conversation_short.wav")
chunk_duration = 30  # seconds
samples_per_chunk = chunk_duration * sample_rate

num_chunks = waveform.size(1) // samples_per_chunk + 1

all_est_sources = []

for i in range(num_chunks):
    start = i * samples_per_chunk
    end = min((i + 1) * samples_per_chunk, waveform.size(1))
    chunk = waveform[:, start:end]

    
    # Skip last chunk if it's shorter than chunk size
    #Yes its dirty but do i care rn?
    if chunk.size(1) < samples_per_chunk:
        print(f"Skipping chunk {i} because it's too short: {chunk.size(1)} samples")
        continue

    chunk_path = f"../output/chunk_{i}.wav"
    torchaudio.save(chunk_path, chunk, sample_rate)
    
    # Separate sources on the chunk
    est_sources_chunk = model.separate_file(path=chunk_path)  # [1, n_src, samples]

    all_est_sources.append(est_sources_chunk)

# Concatenate separated chunks along the time dimension
est_sources = torch.cat(all_est_sources, dim=2)  # dim=2 is time axis

torchaudio.save("..\output\speechbrain/speaker01_lots.wav", est_sources[:, :, 0].detach().cpu(), 16000)
torchaudio.save("..\output\speechbrain/speaker02_less.wav", est_sources[:, :, 1].detach().cpu(), 16000)


Running on: cuda
Skipping chunk 4 because it's too short: 347936 samples


In [ ]:
import torchaudio

waveform, sample_rate = torchaudio.load("../output/speechbrain/conversation_short.wav")
print("Original sample rate:", sample_rate)

chunk_duration = 30  # seconds
samples_per_chunk = chunk_duration * sample_rate

num_chunks = waveform.size(1) // samples_per_chunk + 1

chunk_paths = []

for i in range(num_chunks):
    start = i * samples_per_chunk
    end = min((i + 1) * samples_per_chunk, waveform.size(1))
    chunk = waveform[:, start:end]

    # Skip last chunk if it's too short
    if chunk.size(1) < samples_per_chunk:
        print(f"Skipping chunk {i} because it's too short: {chunk.size(1)} samples")
        continue

    chunk_path = f"../output/speechbrain/chunk_{i}.wav"
    torchaudio.save(chunk_path, chunk, sample_rate)
    chunk_paths.append(chunk_path)

print(f"Saved {len(chunk_paths)} chunks for separation.")


Original sample rate: 16000
Skipping chunk 4 because it's too short: 347936 samples
Saved 4 chunks for separation.


In [ ]:
import torch
from speechbrain.inference import SepformerSeparation as separator
import torchaudio

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Running on:", device)

model = separator.from_hparams(
    source="speechbrain/sepformer-whamr",
    savedir="pretrained_models/sepformer-whamr",
    run_opts={"device": device, "sample_rate": 16000}
)
print("TEst")
all_est_sources = []

for chunk_path in chunk_paths:
    print("TEst02")
    start = time.time()
    est_sources_chunk = model.separate_batch(chunk.to(device))
    end = time.time()

    print(f"Inference took {end - start:.2f} seconds on {device}")

    print("TEst01")
    all_est_sources.append(est_sources_chunk)

# Concatenate separated chunks along time axis (dim=2)
final_sources = torch.cat(all_est_sources, dim=2)

# Save separated speakers as output files
torchaudio.save("../output/speechbrain/speaker01_full.wav", final_sources[:, :, 0].detach().cpu(), sample_rate)
torchaudio.save("../output/speechbrain/speaker02_full.wav", final_sources[:, :, 1].detach().cpu(), sample_rate)

print("Separation done and files saved.")


Running on: cuda
TEst
TEst02
TEst01
TEst02
TEst01
TEst02
TEst01
TEst02
TEst01
Separation done and files saved.
